In [49]:
import os
import json
from datetime import datetime, timezone
from pathlib import Path

from pyspark.context import SparkContext
from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField,
    LongType, IntegerType, DoubleType, TimestampType, StringType, DateType
)

### Configuration, constant paths

In [50]:
BASE_DIR = Path()
INBOX_DIR = BASE_DIR / "data" / "inbox"
OUTBOX_DIR = BASE_DIR / "data" / "outbox"
STATE_DIR = BASE_DIR / "state"
OUTPUT_PATH = OUTBOX_DIR / "trips_enriched.parquet"
BOROUGH_FLOWS_PATH = OUTBOX_DIR / "borough_flows.parquet"
LOOKUP_PATH = BASE_DIR / "data" / "taxi_zone_lookup.parquet"
MANIFEST_PATH = STATE_DIR / "manifest.json"

### State management

In [51]:
def load_manifest() -> dict:
    """Load state from disk or return empty one"""
    if MANIFEST_PATH.exists():
        try:
            with open(MANIFEST_PATH) as file:
                manifest = json.load(file)
            if isinstance(manifest, dict) and "processed_files" in manifest.keys():
                return manifest
        except (json.JSONDecodeError, ValueError):
            pass
    return {"processed_files": {}}

def save_manifest(manifest: dict) -> None:
    """Save state to disk"""
    STATE_DIR.mkdir(parents=True, exist_ok=True)
    tmp = MANIFEST_PATH.with_suffix(".tmp")
    with open(tmp, "w") as file:
        json.dump(manifest, file, indent=2, default=str)
    tmp.replace(MANIFEST_PATH)
    print("Manifest saved to", MANIFEST_PATH)

def file_fingerprint(path: Path) -> dict:
    stat = path.stat()
    return {
        "filename": path.name,
        "filepath": str(path),
        "size_bytes": stat.st_size,
        "mtime": stat.st_mtime
    }

def new_files(inbox: Path, manifest: dict) -> list[Path]:
    processed = set(manifest["processed_files"].keys())
    candidates = sorted(inbox.glob("*.parquet"))
    fresh = [p for p in candidates if p.name not in processed]
    print(f"Found {len(candidates)} parquet files in inbox, {len(fresh)} are new")
    return fresh


### Ingestion

In [52]:
def read_trip_files(spark: SparkSession, paths: list[Path]) -> DataFrame:
    if not paths:
        raise ValueError("Paths list is empty")
    
    str_paths = [str(p) for p in paths]
    df = (spark.read
          .option("mergeSchema", "true")
          .parquet(*str_paths))

    df = df.withColumn("source_file", F.element_at(F.split(F.input_file_name(), "/"), -1))
    print("Raw schema:", df.schema.simpleString())
    return df

def read_zone_lookup(spark: SparkSession) -> DataFrame:
    return spark.read.parquet(str(LOOKUP_PATH))

### Casting and normalization

Cast columns to correct types.

- Parse ISO-8601 UTC timestamps
- Cast numeric fields
- Standardize timestamp column names to: pickup_ts, dropoff_ts

In [53]:
def cast_and_normalize_types(df: DataFrame) -> DataFrame:

    return (
        df
        # Timestamps
        .withColumn("pickup_ts", F.to_timestamp("tpep_pickup_datetime"))
        .withColumn("dropoff_ts", F.to_timestamp("tpep_dropoff_datetime"))

        # Integers
        .withColumn("VendorID", F.col("VendorID").cast(IntegerType()))
        .withColumn("passenger_count", F.col("passenger_count").cast(IntegerType()))
        .withColumn("RatecodeID", F.col("RatecodeID").cast(IntegerType()))
        .withColumn("PULocationID", F.col("PULocationID").cast(IntegerType()))
        .withColumn("DOLocationID", F.col("DOLocationID").cast(IntegerType()))
        .withColumn("payment_type", F.col("payment_type").cast(IntegerType()))

        # Doubles
        .withColumn("trip_distance", F.col("trip_distance").cast(DoubleType()))
        .withColumn("fare_amount", F.col("fare_amount").cast(DoubleType()))
        .withColumn("extra", F.col("extra").cast(DoubleType()))
        .withColumn("mta_tax", F.col("mta_tax").cast(DoubleType()))
        .withColumn("tip_amount", F.col("tip_amount").cast(DoubleType()))
        .withColumn("tolls_amount", F.col("tolls_amount").cast(DoubleType()))
        .withColumn("improvement_surcharge", F.col("improvement_surcharge").cast(DoubleType()))
        .withColumn("total_amount", F.col("total_amount").cast(DoubleType()))
        .withColumn("congestion_surcharge", F.col("congestion_surcharge").cast(DoubleType()))
        .withColumn("Airport_fee", F.col("Airport_fee").cast(DoubleType()))
        .withColumn("cbd_congestion_fee", F.col("cbd_congestion_fee").cast(DoubleType()))
    )

### Cleaning

Apply the following data quality rules:

1. Drop rows with critical nulls: pickup_ts, dropoff_ts, PULocationID, DOLocationID, total_amount
2. dropoff_ts must be > pickup_ts
3. passenger_count valid range: 1–8 (otherwise NULL)
4. trip_distance must be > 0
5. Monetary fields must be >= 0 (otherwise NULL)

In [54]:
def clean_trip_data(df: DataFrame) -> DataFrame:
    # Drop critical nulls
    df = df.dropna(subset=[
        "pickup_ts",
        "dropoff_ts",
        "PULocationID",
        "DOLocationID",
        "total_amount"
    ])

    # Valid trip duration
    df = df.filter(F.col("dropoff_ts") > F.col("pickup_ts"))

    # Passenger count rule
    df = df.withColumn(
        "passenger_count",
        F.when(
            (F.col("passenger_count") >= 1) &
            (F.col("passenger_count") <= 8),
            F.col("passenger_count")
        ).otherwise(None)
    )

    # Trip distance rule
    df = df.filter(F.col("trip_distance") > 0)

    # Monetary validation
    money_cols = [
        "fare_amount", "extra", "mta_tax", "tip_amount",
        "tolls_amount", "improvement_surcharge",
        "total_amount", "congestion_surcharge",
        "Airport_fee", "cbd_congestion_fee"
    ]

    for c in money_cols:
        df = df.withColumn(
            c,
            F.when(F.col(c) >= 0, F.col(c)).otherwise(None)
        )

    return df

### Deduplication
Deduplicate using business key:

(VendorID,
 pickup_ts,
 dropoff_ts,
 PULocationID,
 DOLocationID)

These fields uniquely identify a taxi trip event.


In [55]:
def deduplicate_trips(df: DataFrame) -> DataFrame:

    dedup_key = [
        "VendorID",
        "pickup_ts",
        "dropoff_ts",
        "PULocationID",
        "DOLocationID",
    ]

    return df.dropDuplicates(dedup_key)

### Enrichment
1. Joins with zones lookup for both Pickup and Dropoff locations.
2. Calculates trip_duration_minutes and pickup_date.
3. Adds ingestion metadata.

In [56]:
def enrich_trip_data(df: DataFrame, zones: DataFrame, ingested_at: str) -> DataFrame:
    pickup_lookup = F.broadcast(
        zones.select(
            F.col("LocationID").alias("pickup_zone_id"),
            F.col("Zone").alias("pickup_zone_name"),
            F.col("Borough").alias("pickup_borough")
        )
    )

    dropoff_lookup = F.broadcast(
        zones.select(
            F.col("LocationID").alias("dropoff_zone_id"),
            F.col("Zone").alias("dropoff_zone_name"),
            F.col("Borough").alias("dropoff_borough")
        )
    )

    # Join for pickup lookup
    df = df.join(
        pickup_lookup, 
        df.PULocationID == pickup_lookup.pickup_zone_id, 
        how="left"
    ).drop("pickup_zone_id")

    # Join for dropoff lookup
    df = df.join(
        dropoff_lookup, 
        df.DOLocationID == dropoff_lookup.dropoff_zone_id, 
        how="left"
    ).drop("dropoff_zone_id")

    # Derived Columns
    df = (
        df
        # duration = (dropoff - pickup) / 60 seconds
        .withColumn("trip_duration_minutes", 
            (F.unix_timestamp("dropoff_ts") - F.unix_timestamp("pickup_ts")) / 60.0
        )
        .withColumn("pickup_date", F.to_date("pickup_ts"))
        .withColumn("ingested_at", F.lit(ingested_at))
    )

    return df

### Custom scenario
For each month, we calculate how many trips occurred between each pair of boroughs and what was the average trip distance.

In [57]:
def compute_borough_flows(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("pickup_year_month", F.date_format("pickup_ts", "yyyy-MM"))
        .groupBy("pickup_borough", "dropoff_borough", "pickup_year_month")
        .agg(
            F.count("*").alias("trip_count"),
            F.avg("trip_distance").alias("avg_trip_distance")
        )
        .orderBy("pickup_year_month", F.desc("trip_count"))
    )

### Main method for whole process

In [58]:
def main():
    
    INBOX_DIR.mkdir(parents=True, exist_ok=True)
    OUTBOX_DIR.mkdir(parents=True, exist_ok=True)
    STATE_DIR.mkdir(parents=True, exist_ok=True)

    manifest = load_manifest()
    print(f"Manifest loaded, {len(manifest["processed_files"])} files previously processed")

    files_to_process = new_files(INBOX_DIR, manifest)

    if not files_to_process:
        print("No new files found, exiting")
        return
    
    ingested_at = datetime.now(timezone.utc).isoformat()

    sc = SparkContext('local', 'project_01')
    spark = (
        SparkSession.builder
        .appName('project_01')
        .enableHiveSupport()
        .getOrCreate()
    )
    print(spark.version)

    try:
        # Ingest
        zones = read_zone_lookup(spark)
        print("Zone lookup count:", zones.count())
        raw_df = read_trip_files(spark, files_to_process)
        print("Ingested raw data with schema:", raw_df.schema.simpleString())
        # Cast types
        df = raw_df
        df = cast_and_normalize_types(raw_df)
        print("After casting, schema is:", df.schema.simpleString())
        # Clean
        print("Before cleaning, count is:", df.count())
        df = clean_trip_data(df)
        print("After cleaning, count is:", df.count())
        # Deduplicate
        df = deduplicate_trips(df)
        print("After deduplication, count is:", df.count())
        # Enrich with zones
        df = enrich_trip_data(df, zones, ingested_at)

        if OUTPUT_PATH.exists():
            print("Existing output found. Merging...")
            existing_df = spark.read.parquet(str(OUTPUT_PATH))
            # 'allowMissingColumns=True' for added borough columns
            full_df = existing_df.unionByName(df, allowMissingColumns=True)
            full_df = deduplicate_trips(full_df)
        else:
            print("No existing output. Creating new baseline.")
            full_df = df

        full_df = full_df.cache()
        total_rows = full_df.count()

        (full_df.coalesce(1).write
         .mode("overwrite")
         .parquet(str(OUTPUT_PATH)))

        borough_flows_df = compute_borough_flows(full_df)
        (borough_flows_df.coalesce(1).write
         .mode("overwrite")
         .parquet(str(BOROUGH_FLOWS_PATH)))

        #full_df.unpersist()

        # Update manifest
        per_file_counts = {
            row["source_file"]: row["count"]
            for row in df.groupBy("source_file").count().collect()
        }
        for p in files_to_process:
            fp = file_fingerprint(p)
            fp["row_count"] = per_file_counts.get(p.name, 0)
            fp["processed_at"] = ingested_at
            manifest["processed_files"][p.name] = fp

        manifest["last_run"] = ingested_at
        manifest["total_output_rows"] = total_rows
        save_manifest(manifest)

    finally:
        spark.stop()
        

### Run it

In [59]:
main()

Manifest loaded, 2 files previously processed
Found 2 parquet files in inbox, 0 are new
No new files found, exiting


### See what's in the result files

In [60]:
spark = SparkSession.builder.appName('inspection').getOrCreate()

df_check = spark.read.parquet(str(OUTPUT_PATH))

df_check.printSchema()

df_check.show(5, truncate=False)

print(f"Total rows in output: {df_check.count()}")



flows_df = spark.read.parquet(str(BOROUGH_FLOWS_PATH))

flows_df.printSchema()

flows_df.orderBy(F.desc("trip_count")).show(10, truncate=False)
print(f"Total borough flow rows: {flows_df.count()}")

spark.stop()


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- pickup_ts: timestam